### Milestone 1 with New Corpus 

In [30]:
import pandas as pd
import time

import matplotlib.pyplot as plt
import json
import numpy as np
from pathlib import Path

from langchain_community.callbacks import get_openai_callback
from langchain_classic.chains import create_retrieval_chain
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

from langchain_chroma import Chroma
import re


from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import BSHTMLLoader

import hashlib
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_text_splitters import HTMLHeaderTextSplitter, RecursiveCharacterTextSplitter
from pathlib import Path

import random
from pydantic import BaseModel, Field
from collections import defaultdict
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import Optional, Dict, List
import tiktoken
from langchain_classic.chains import RetrievalQA

random.seed(42) # I sure hope this works! 

In [8]:
import os
from dotenv import load_dotenv

load_dotenv()

# ── UNC AI Gateway (Azure API Management in front of Azure OpenAI — OpenAI-compatible v1 API) ──
# One key, one endpoint, every course model. Your personal key arrives via Canvas Inbox.
# Create a .env file next to this notebook containing a single line:
# See .env.example for the required API key configuration.
# ⚠️  Never paste your key into a notebook cell — notebooks get submitted!
UNC_AI_BASE_URL = "https://azureaiapi.cloud.unc.edu/openai/v1"
UNC_AI_API_KEY = os.getenv("UNC_AI_API_KEY") or os.getenv("OPENAI_API_KEY")
assert UNC_AI_API_KEY, "Missing UNC_AI_API_KEY — check the Canvas Inbox message with your personal key"

# Course model lineup (deployment names on the gateway match the model names)
CHAT_MODEL = os.getenv("UNC_CHAT_MODEL", "gpt-4.1-mini")        # everyday workhorse
ADVANCED_MODEL = os.getenv("UNC_ADVANCED_MODEL", "gpt-5.6-sol")  # frontier tier for harder tasks
EMBED_MODEL = os.getenv("UNC_EMBED_MODEL", "text-embedding-3-small")

# Let every OpenAI-compatible library (LangChain, LlamaIndex, CrewAI, AutoGen…) find the gateway:
os.environ["OPENAI_API_KEY"] = UNC_AI_API_KEY
os.environ["OPENAI_BASE_URL"] = UNC_AI_BASE_URL

# Gateway policy: if you ever use the Responses API, set store=False and never background=True.

print("✓ UNC AI Gateway configured")
print(f"  Chat: {CHAT_MODEL}  |  Advanced: {ADVANCED_MODEL}  |  Embeddings: {EMBED_MODEL}")

✓ UNC AI Gateway configured
  Chat: gpt-4.1-mini  |  Advanced: gpt-5.6-sol  |  Embeddings: text-embedding-3-small


In [9]:
print(f"Gateway: {UNC_AI_BASE_URL}")
print(f"Embedding model: {EMBED_MODEL}")

# Embeddings through the UNC AI Gateway
try:
    embeddings = OpenAIEmbeddings(
        model=EMBED_MODEL,
        base_url=UNC_AI_BASE_URL,
        api_key=UNC_AI_API_KEY,
    )

    # Test with a simple embedding
    print("Testing embedding with single text...")
    test_embedding = embeddings.embed_query("test")
    print(f"✓ Embedding successful! Vector dimension: {len(test_embedding)}")

except Exception as e:
    print(f"✗ Embedding failed: {e}")
    print("\nTroubleshooting:")
    print("1. Confirm UNC_AI_API_KEY is set in your .env file")
    print("2. Verify your key works: it was sent to you via Canvas Inbox")
    print("3. Check the model name — see the course model lineup in the setup cell")
    raise

print("✓ Embeddings client ready")

Gateway: https://azureaiapi.cloud.unc.edu/openai/v1
Embedding model: text-embedding-3-small
Testing embedding with single text...
✓ Embedding successful! Vector dimension: 1536
✓ Embeddings client ready


In [10]:
# Chat model through the UNC AI Gateway
llm = ChatOpenAI(
    model=CHAT_MODEL, # 4.1 mini is default 
    base_url=UNC_AI_BASE_URL,
    api_key=UNC_AI_API_KEY,
    temperature=0,
)

prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:

{context}

Question: {input}"""
)

In [11]:
def sanitize_metadata(metadata: dict) -> dict:
    clean = {}
    for key, value in metadata.items():
        new_key = re.sub(r'[^_0-9A-Za-z]', '_', key)
        # Weaviate also requires the first char to be a letter or underscore
        if new_key[0].isdigit():
            new_key = f"_{new_key}"
        clean[new_key] = value
    return clean


In [79]:
def ask_question(question: str):

    response = qa_chain_large.invoke({"query": question})

    print(f"Q: {question}")
    print("=" * 60)
    print("ANSWER:")
    print(response["result"])
    print("=" * 60)

    sources = response["source_documents"]
    print(f"\n[Retrieved {len(sources)} sources]")

    for i, doc in enumerate(sources, 1):
        print(
            f"\nSource {i}: "
            f"{doc.metadata.get('source', 'N/A')} "
            f"(Page {doc.metadata.get('page', 'N/A')})"
        )

    return response

In [13]:
HEADERS = [("h1", "h1"), ("h2", "h2"), ("h3", "h3")]
html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=HEADERS)
char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, chunk_overlap=150, add_start_index=True  # match your existing settings
)

def load_htm(path: str) -> list[Document]:
    raw = Path(path).read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(raw, "html.parser")

    title = soup.title.get_text(strip=True) if soup.title else ""

    for tag in soup(["script", "style", "noscript", "iframe", "svg", "nav", "footer", "header", "form"]):
        tag.decompose()
    cleaned = str(soup)

    sections = html_splitter.split_text(cleaned)
    docs = []
    for i, s in enumerate(sections):
        section = " > ".join(s.metadata[k] for k in ("h1", "h2", "h3") if k in s.metadata)
        docs.append(Document(
            page_content=s.page_content,
            metadata={"source": path, "title": title,
                      "section": section or f"section_{i}", "section_idx": i},
        ))
    return docs

def add_chunk_ids(chunks: list[Document]) -> list[Document]:
    for c in chunks:
        m = c.metadata
        key = f"{m.get('source')}|{m.get('page')}|{m.get('row')}|{m.get('section_idx')}|{m.get('start_index')}"
        m["chunk_id"] = hashlib.sha1(key.encode()).hexdigest()[:12]
    return chunks

In [14]:
def load_document(file_path):
    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix == ".pdf":
        docs = PyPDFLoader(str(path)).load()      # metadata: source, page (0-indexed)
        ftype = "pdf"
        
    elif suffix == ".csv":
        docs = CSVLoader(str(path)).load()        # metadata: source, row
        ftype = "csv"
        
    elif suffix in (".htm", ".html"):
        docs = load_htm(str(path))                # metadata: source, title, section, section_idx
        ftype = "html"

    elif suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            article = json.load(f)

        docs = [
            Document(
                page_content=article["text"],
                metadata={
                    "source": str(path),
                    "title": article["title"],
                    "source_url": article["source"],
                    "revision_id": article["revision_id"],
                    "pageid": article["pageid"]
                }
            )
        ]
        ftype = "json"
        
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    for d in docs:
        d.metadata["file_type"] = ftype
        
    return docs

In [17]:
data_dir = Path("corpus")

files = list(data_dir.rglob("*.pdf"))
files += list(data_dir.rglob("*.htm"))
files += list(data_dir.rglob("*.csv"))
files += list(data_dir.rglob("*.json"))

print(f"Found {len(files)} files")

Found 85 files


In [18]:
all_documents = []

for file in files:
    docs = load_document(file)
    all_documents.extend(docs)

    # print(f"{file.name}: {len(docs)} documents")

print(f"Total documents loaded: {len(all_documents)}")    

Total documents loaded: 1941


In [19]:
# check to make sure this is actually what I wanted! 


print(f"\nPreview ({len(all_documents[0].page_content)} characters):")
print(all_documents[0].page_content[:500] + "...")
print("\n")

print(f"\nPreview ({len(all_documents[10].page_content)} characters):")
print(all_documents[8].page_content[:600] + "...")
print("\n")

print(f"\nPreview ({len(all_documents[20].page_content)} characters):")
print(all_documents[8].page_content[:600] + "...")
print("\n")

print(f"\nPreview ({len(all_documents[40].page_content)} characters):")
print(all_documents[8].page_content[:600] + "...")
print("\n")

print(f"\nPreview ({len(all_documents[60].page_content)} characters):")
print(all_documents[8].page_content[:600] + "...")
print("\n")


print(f"\nPreview ({len(all_documents[-1].page_content)} characters):")
print(all_documents[-1].page_content[:500] + "...")



Preview (7673 characters):
HORTSCIENCE 50(11):1610–1617. 2015.
The Carbohydrate Yield of Sweetpotato
(Ipomoea batatas) Grown from Slips
and Root Pieces in North Carolina
Nicholas A. George
Department of Horticultural Science, North Carolina State University, 214
Kilgore Hall, Campus Box 7609, Raleigh, NC 27695; and Department of
Plant Sciences, the University of California, Davis, One Shields Avenue
Davis, CA 95616
Kenneth V. Pecota and G. Craig Yencho 1
Department of Horticultural Science, North Carolina State University...



Preview (8831 characters):
Genetic Diversity, Fungicide Sensitivity, and Host Resistance
to Ceratocystis fimbriata Infecting Sweetpotato in North Carolina
A. C. Scruggs, Department of Plant Pathology, North Carolina State University, Raleigh 27695; T. Basaiah, Department of Microbiology,
Kuvempu University, Shimoga, Karnataka, India; and M. L. Adams and L. M. Quesada-Ocampo, Department of Plant Pathology, North
Carolina State University, Raleigh 27695
Abstract


In [26]:
# NOT DEAD CODE!!!

# # for reruns
# # check these first...if error, then you're good! they haven't been made yet
# print(vectorstore_large._collection.count())

# # if those do run, then do these to remove them
# vectorstore_large.delete_collection()

In [27]:
splitter_large = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks_large = splitter_large.split_documents(all_documents)
print(f"Large chunks (1000 chars): {len(chunks_large)} chunks")

Large chunks (1000 chars): 5149 chunks


In [28]:
# Create the vector store for the large chunk strategy
print(f"Embedding {len(chunks_large)} chunks...")

vectorstore_large = Chroma.from_documents(
    documents=chunks_large,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="rag_large"
)

print(f"\n✓ Vector store created with {vectorstore_large._collection.count()} vectors")

Embedding 5149 chunks...

✓ Vector store created with 5149 vectors


In [31]:
qa_chain_large = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore_large.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True
)

In [37]:
def make_key(source, location):
    return (
        Path(source).name,
        tuple(sorted(location.items()))
    )

In [41]:
def location_of(doc):
    meta = doc.metadata
    suffix = Path(meta.get("source", "")).suffix.lower()

    if suffix == ".pdf":
        page = meta.get("page")
        return {"page": page} if page is not None else {}

    if suffix == ".csv":
        row = meta.get("row")
        return {"row": row} if row is not None else {}

    if suffix in (".htm", ".html"):
        idx = meta.get("section_idx")
        return {"section_idx": idx} if idx is not None else {}

    if suffix == ".json":
        chunk_id = meta.get("chunk_id")
        return {"chunk_id": chunk_id} if chunk_id is not None else {}
        
    return {}

In [45]:
def evaluate_retrieval_strategies(gold_dataset, retrievers_dict):
    rows = []

    for strategy_name, retriever in retrievers_dict.items():

        for item in gold_dataset:

            # What sources/locations should have been retrieved?
            expected_keys = {
                make_key(
                    expected["source"],
                    {
                        k: v
                        for k, v in expected.items()
                        if k != "source"
                    }
                )
                for expected in item["expected_sources"]
            }

            # What did the retriever actually return?
            retrieved_docs = retriever.invoke(item["query"])

            retrieved_keys = [
                make_key(
                    doc.metadata.get("source", ""),
                    location_of(doc)
                )
                for doc in retrieved_docs
            ]

            # Rank of first correct result
            first_rank = next(
                (
                    rank
                    for rank, key in enumerate(retrieved_keys, start=1)
                    if key in expected_keys
                ),
                None
            )

            # Metrics for this question
            hit = int(first_rank is not None)

            reciprocal_rank = (
                1 / first_rank
                if first_rank is not None
                else 0.0
            )

            recall = (
                len(expected_keys & set(retrieved_keys))
                / len(expected_keys)
                if expected_keys
                else 0.0
            )

            precision = (
                sum(key in expected_keys for key in retrieved_keys)
                / len(retrieved_keys)
                if retrieved_keys
                else 0.0
            )

            rows.append({
                "strategy": strategy_name,
                "filetype": (
                    Path(item["expected_sources"][0]["source"]).suffix.lower().lstrip(".")
                    if item["expected_sources"]
                    else "unanswerable"
                ),
                "query": item["query"],
                "hit": hit,
                "reciprocal_rank": reciprocal_rank,
                "recall": recall,
                "precision": precision,
                "expected_sources": item["expected_sources"],

                    # Keep track of what the retriever actually returned
                "retrieved_sources": [
                    {
                        "source": Path(doc.metadata.get("source", "")).name,
                        "page": doc.metadata.get("page"),
                        "row": doc.metadata.get("row"),
                        "section_idx": doc.metadata.get("section_idx")
                    }
                    for doc in retrieved_docs
                ]
            })

    return pd.DataFrame(rows)


In [52]:
with open("gold_set_m2.json", "r", encoding="utf-8") as f:
    gold_dataset = json.load(f)

print("Original gold questions:", len(gold_dataset))

Original gold questions: 68


In [53]:
retrievers = {
    "Baseline": vectorstore_large.as_retriever(search_kwargs={"k": 3}),
    }

answerable_gold = [
    item for item in gold_dataset
    if item["expected_sources"]
]

per_q = evaluate_retrieval_strategies(answerable_gold,retrievers)

In [54]:
# This was the same intention as my Lab 4 output, which I pasted into ChatGPT to get help for updating with the multiple file types
overall = (
    per_q
    .groupby("strategy")
    .agg(
        n=("hit", "count"),
        hit_rate=("hit", "mean"),
        recall=("recall", "mean"),
        mrr=("reciprocal_rank", "mean"),
        precision=("precision", "mean"),
    )
    .round(3)
)

print(overall)

           n  hit_rate  recall    mrr  precision
strategy                                        
Baseline  62     0.742   0.742  0.675       0.36


In [56]:
by_type = (
    per_q
    .groupby(["filetype", "strategy"])
    .agg(
        n=("hit", "count"),
        hit_rate=("hit", "mean"),
        recall=("recall", "mean"),
        mrr=("reciprocal_rank", "mean"),
        precision=("precision", "mean"),
    )
    .round(3)
)

print(by_type)

                    n  hit_rate  recall    mrr  precision
filetype strategy                                        
csv      Baseline  11     0.545   0.545  0.545      0.182
htm      Baseline  10     0.700   0.700  0.633      0.300
json     Baseline  16     0.875   0.875  0.802      0.604
pdf      Baseline  25     0.760   0.760  0.667      0.307


In [57]:
per_q.to_csv("baseline_retrieval_m2.csv", index=False)
by_type.to_csv("baseline_retrieval_m2_by_type.csv", index=False)

In [58]:
# the judge LLM
judge_llm = ChatOpenAI(
    model=ADVANCED_MODEL, # gpt-5.6-sol is default now
    base_url=UNC_AI_BASE_URL,
    api_key=UNC_AI_API_KEY,
    use_responses_api=False
)


JUDGE_PROMPT = """You are an expert evaluator assessing a RAG system.

The retrieved context is a set of labeled chunks. Chunks may be PDF text, HTML text,
or CSV rows formatted as "column: value" lines. Treat values in CSV rows as valid evidence.

Question: {question}

Retrieved Context:
{context}

Generated Answer: {answer}

Score each item as an integer from 1 to 5:

1. "faithfulness": Is every claim in the generated answer supported by the retrieved context?
   (1 = mostly unsupported or invented, 5 = fully supported).
   If the answer says the information is not available and it truly is not in the context,
   score 5.

2. "relevance": Does the generated answer directly address the question?
   (1 = off-topic, 5 = fully answers it)

3. "reason": One sentence explaining the scores.

Respond ONLY with a JSON object with the keys:
faithfulness, relevance, reason.
"""

SCORE_KEYS = ["faithfulness", "relevance"]


# This function is from Claude, metadata across file types is hard!
def format_context(docs, max_chars=3000):
    parts = []
    for i, d in enumerate(docs, 1):
        loc = ", ".join(f"{k} {v}" for k, v in location_of(d).items())
        name = Path(d.metadata.get("source", "")).name
        text = unicodedata.normalize("NFKC", d.page_content).strip()[:max_chars]
        parts.append(f"[Chunk {i} | {name} | {loc}]\n{text}")
    return "\n\n".join(parts)


# This function is from Claude
def parse_judge_json(text):
    """Tolerates code fences and extra prose; returns None if unusable."""
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        data = json.loads(match.group(0))
        return {
            **{k: int(data[k]) for k in SCORE_KEYS},
            "reason": str(data.get("reason", "")),
        }
    except (json.JSONDecodeError, KeyError, ValueError, TypeError):
        return None


In [59]:
def evaluate_single_generation(question, retrieved_docs, answer, judge_llm):
    """
    sends a generated answer and its context to the LLM judge for scoring
    """
    # combine retrieved document content into a single context string
    context_text = ""
    for doc in retrieved_docs:
        context_text = context_text + doc.page_content + "\n\n"

    # format the judge prompt
    formatted_prompt = JUDGE_PROMPT.format(
        question=question,
        context=context_text,
        answer=answer
    )

    response = judge_llm.invoke(formatted_prompt)
    
    # this next part was with AI help
    try:
        scores = json.loads(response.content)
    except json.JSONDecodeError:
        scores = {
            "faithfulness": 0,
            "relevance": 0,
            "correctness": 0,
            "reason": "Failed to parse JSON response from judge."
        }

    return scores

In [60]:
def evaluate_generation_quality(gold_dataset, qa_chains_dict, judge_llm):
    """
    evaluates generation quality across multiple RAG strategies using an LLM judge
    """
    summary_results = []

    for strategy_name, qa_chain in qa_chains_dict.items():
        # set scores
        total_faithfulness = 0
        total_relevance = 0
        total_queries = len(gold_dataset)

        for item in gold_dataset:
            query = item["query"]

            # do the RAG like in ask_question function
            response = qa_chain.invoke({"query": query})
            generated_answer = response["result"]
            retrieved_docs = response["source_documents"]

            # judge the answer 
            evaluation = evaluate_single_generation(
                question=query,
                retrieved_docs=retrieved_docs,
                answer=generated_answer,
                judge_llm=judge_llm
            )

            # update the scores
            total_faithfulness = total_faithfulness + evaluation["faithfulness"]
            total_relevance = total_relevance + evaluation["relevance"]

        # calculate averages 
        avg_faithfulness = total_faithfulness / total_queries
        avg_relevance = total_relevance / total_queries

        summary_results.append({
            "Strategy": strategy_name,
            "Avg Faithfulness (1-5)": round(avg_faithfulness, 3),
            "Avg Relevance (1-5)": round(avg_relevance, 3)
        })

    return pd.DataFrame(summary_results)

In [61]:
def evaluate_generation(gold_dataset, vectorstore, llm, judge_llm, k=3):
    """
    Evaluate generated answers using an LLM judge.

    Scores:
    - Faithfulness: 1-5
    - Relevance: 1-5
    """

    results = []
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )

    total_queries = len(gold_dataset)

    for i, item in enumerate(gold_dataset, start=1):

        # print(f"Evaluating question {i}/{total_queries}...")

        query = item["query"]

        response = qa_chain.invoke({"query": query})

        generated_answer = response["result"]
        retrieved_docs = response["source_documents"]

        eval_scores = evaluate_single_generation(
            question=query,
            retrieved_docs=retrieved_docs,
            answer=generated_answer,
            judge_llm=judge_llm
        )

        results.append({
            "query": query,
            "answer": generated_answer,
            "faithfulness": eval_scores.get("faithfulness", 0),
            "relevance": eval_scores.get("relevance", 0),
            "reason": eval_scores.get("reason", ""),
            "retrieved_docs": retrieved_docs
        })

    return pd.DataFrame(results)

In [62]:
judge_results = evaluate_generation(answerable_gold, vectorstore_large, llm, judge_llm, k=3)

In [64]:
# print(judge_results)

In [65]:
print("Mean Faithfulness:",
      judge_results["faithfulness"].mean())

print("Mean Relevance:",
      judge_results["relevance"].mean())

Mean Faithfulness: 4.629032258064516
Mean Relevance: 4.903225806451613


### Unanswerable questions

In [66]:
unanswerable_gold = [
    item for item in gold_dataset
    if not item["expected_sources"]
]

In [80]:
unanswerable_results = []


for item in unanswerable_gold:
    response = ask_question(item["query"])

    unanswerable_results.append({
        "query": item["query"],
        "answer": response["result"],
        "sources": response["source_documents"]
    })

Q: What will be the leading cause of sweet potato crop losses in North Carolina in 2035?
ANSWER:
The provided information does not specify the leading cause of sweet potato crop losses in North Carolina in 2035. It mentions current challenges such as black rot disease, insecticide resistance, biotic and abiotic stress factors, and the impact of climate change on sweet potato production, but it does not predict which will be the leading cause of losses in 2035. Therefore, I do not have enough information to determine the leading cause of sweet potato crop losses in North Carolina in 2035.

[Retrieved 3 sources]

Source 1: corpus\original_files\original_pdfs\rag_sp2.pdf (Page 0)

Source 2: corpus\original_files\original_pdfs\rag_sp4.pdf (Page 8)

Source 3: corpus\new_files\proquest\Biotic_Stress_Resistance_in_Sw.pdf (Page 1)
Q: How many acres of sweet potatoes were harvested in Siler City, North Carolina, in 2025?
ANSWER:
The provided information does not include data on the number of ac